# Space Station Object Detection with YOLOv8

This notebook demonstrates the training, validation, and evaluation of a YOLOv8 model for detecting critical objects (toolbox, oxygen tank, fire extinguisher) in a space station environment.

## Contents
1. Setup and Installation
2. Dataset Preparation
3. Model Training
4. Model Validation
5. Performance Evaluation
6. Visualization & Export

## 1. Setup and Installation

First, let's install the necessary dependencies.

In [ ]:
# Install ultralytics package for YOLOv8
!pip install ultralytics

# Install additional packages for visualization
!pip install opencv-python matplotlib seaborn pandas scikit-learn

In [ ]:
# Import necessary libraries
import os
import sys
import random
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import seaborn as sns
import pandas as pd
from pathlib import Path
from sklearn.metrics import confusion_matrix
from ultralytics import YOLO

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Configure paths to dataset
dataset_root = Path('../dataset')
train_dir = dataset_root / 'Train'
val_dir = dataset_root / 'Val'
test_dir = dataset_root / 'Test'

# Check if dataset directories exist
for dir_path in [train_dir, val_dir, test_dir]:
    if not dir_path.exists():
        print(f"Warning: {dir_path} directory not found. Please make sure your dataset is properly structured.")

## 2. Dataset Preparation

The dataset consists of images of space station interiors with annotations for three object classes:
1. Fire Extinguisher
2. Toolbox
3. Oxygen Tank

The annotations are in YOLO format, with one text file per image containing one line per object in the format:
```
<class_id> <x_center> <y_center> <width> <height>
```

Let's create a dataset configuration file and verify our dataset structure.

In [ ]:
# Define class names
class_names = ['Fire Extinguisher', 'Toolbox', 'Oxygen Tank']

# Create dataset YAML configuration file
yaml_content = f"""
# Space Station Object Detection Dataset
path: {dataset_root.absolute()}  # dataset root directory
train: Train/images  # train images (relative to 'path')
val: Val/images  # val images (relative to 'path')
test: Test/images  # test images (optional)

# Classes
names:
  0: Fire Extinguisher
  1: Toolbox
  2: Oxygen Tank
"""

# Write YAML file
yaml_path = dataset_root / 'space_station_dataset.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"Dataset configuration saved to {yaml_path}")

In [ ]:
# Function to check dataset structure
def check_dataset_structure(dataset_dir):
    images_dir = dataset_dir / 'images'
    labels_dir = dataset_dir / 'labels'
    
    # Create directories if they don't exist
    images_dir.mkdir(exist_ok=True, parents=True)
    labels_dir.mkdir(exist_ok=True, parents=True)
    
    # Count image and label files
    image_files = list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png'))
    label_files = list(labels_dir.glob('*.txt'))
    
    print(f"Found {len(image_files)} images and {len(label_files)} label files in {dataset_dir}.")
    
    return len(image_files), len(label_files)

# Check each dataset split
train_images, train_labels = check_dataset_structure(train_dir)
val_images, val_labels = check_dataset_structure(val_dir)
test_images, test_labels = check_dataset_structure(test_dir)

print(f"\nTotal dataset statistics:")
print(f"Training: {train_images} images")
print(f"Validation: {val_images} images")
print(f"Testing: {test_images} images")
print(f"Total: {train_images + val_images + test_images} images")

## 3. Model Training

Now, let's train a YOLOv8 model on our dataset. We'll start with a pre-trained model and fine-tune it on our space station object detection task.

In [ ]:
# Load a pre-trained YOLOv8 model
model = YOLO('yolov8n.pt')  # nano model, you can use 's', 'm', 'l', or 'x' for larger models

# Set training parameters
epochs = 100
batch_size = 16
imgsz = 640
patience = 20  # Early stopping patience

print(f"Starting training for {epochs} epochs...")

In [ ]:
# Train the model
results = model.train(
    data=yaml_path,
    epochs=epochs,
    batch=batch_size,
    imgsz=imgsz,
    patience=patience,
    save=True,
    device='0',  # Use '0' for the first GPU, 'cpu' for CPU
    workers=4,
    pretrained=True,
    optimizer='Adam',
    lr0=0.001,
    cos_lr=True,
    weight_decay=0.0005,
    label_smoothing=0.1,
    project='../backend/model',
    name='space_station_detector'
)

## 4. Model Validation

After training, let's validate the model on our validation dataset to assess its performance.

In [ ]:
# Validate the model
val_results = model.val(data=yaml_path, split='val')

# Print validation metrics
print("\nValidation Results:")
print(f"mAP@50: {val_results.box.map50:.4f}")
print(f"mAP@50-95: {val_results.box.map:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall: {val_results.box.mr:.4f}")

## 5. Performance Evaluation

Now, let's evaluate the model on the test set and calculate detailed performance metrics for each class.

In [ ]:
# Evaluate on test set
test_results = model.val(data=yaml_path, split='test')

# Print test metrics
print("\nTest Results:")
print(f"mAP@50: {test_results.box.map50:.4f}")
print(f"mAP@50-95: {test_results.box.map:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall: {test_results.box.mr:.4f}")

In [ ]:
# Extract class-wise metrics
class_metrics = []
for i, class_name in enumerate(class_names):
    class_metric = {
        'className': class_name,
        'precision': test_results.box.mp_per_class[i] if i < len(test_results.box.mp_per_class) else 0,
        'recall': test_results.box.mr_per_class[i] if i < len(test_results.box.mr_per_class) else 0,
        'mAP': test_results.box.map50_per_class[i] if i < len(test_results.box.map50_per_class) else 0
    }
    # Calculate F1 score
    class_metric['f1Score'] = 2 * (class_metric['precision'] * class_metric['recall']) / (class_metric['precision'] + class_metric['recall']) if (class_metric['precision'] + class_metric['recall']) > 0 else 0
    class_metrics.append(class_metric)

# Create a DataFrame for better visualization
metrics_df = pd.DataFrame(class_metrics)
metrics_df = metrics_df.round(4)
display(metrics_df)

In [ ]:
# Visualize class-wise metrics
plt.figure(figsize=(10, 6))
bar_width = 0.2
index = np.arange(len(class_names))

plt.bar(index, metrics_df['precision'], bar_width, label='Precision')
plt.bar(index + bar_width, metrics_df['recall'], bar_width, label='Recall')
plt.bar(index + 2*bar_width, metrics_df['f1Score'], bar_width, label='F1-Score')
plt.bar(index + 3*bar_width, metrics_df['mAP'], bar_width, label='mAP@0.5')

plt.xlabel('Class')
plt.ylabel('Score')
plt.title('Performance Metrics by Class')
plt.xticks(index + 1.5*bar_width, class_names)
plt.ylim(0, 1.0)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('../backend/model/class_performance.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Confusion Matrix and Error Analysis

In [ ]:
# Get confusion matrix
confusion_matrix = test_results.confusion_matrix.matrix

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix, annot=True, fmt='.0f', cmap='Blues',
            xticklabels=class_names + ['Background'], 
            yticklabels=class_names + ['Background'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('../backend/model/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Failure Analysis

Let's identify and analyze some of the model's failure cases.

In [ ]:
# Run inference on test set and collect errors
failure_cases = []
failure_images = []

# Get test image paths
test_images_dir = test_dir / 'images'
test_image_paths = list(test_images_dir.glob('*.jpg')) + list(test_images_dir.glob('*.png'))

# Run through a subset of test images to find failures
for img_path in test_image_paths[:50]:  # Limit to first 50 images for demonstration
    # Get corresponding label file
    label_path = test_dir / 'labels' / (img_path.stem + '.txt')
    
    if not label_path.exists():
        continue
        
    # Read ground truth labels
    gt_boxes = []
    with open(label_path, 'r') as f:
        for line in f:
            cls_id, x_center, y_center, width, height = map(float, line.strip().split())
            gt_boxes.append((int(cls_id), (x_center, y_center, width, height)))
    
    # Run prediction
    results = model(img_path, conf=0.25)
    
    # Check if there are significant mismatches
    if len(gt_boxes) != len(results[0].boxes):
        failure_cases.append({
            'image': img_path.name,
            'ground_truth_count': len(gt_boxes),
            'prediction_count': len(results[0].boxes),
            'error_type': 'Count mismatch'
        })
        failure_images.append(img_path)
    
    # Check for class mismatches
    if len(gt_boxes) > 0 and len(results[0].boxes) > 0:
        gt_classes = [cls for cls, _ in gt_boxes]
        pred_classes = results[0].boxes.cls.tolist()
        
        if sorted(gt_classes) != sorted([int(c) for c in pred_classes]):
            failure_cases.append({
                'image': img_path.name,
                'ground_truth_classes': [class_names[int(cls)] for cls in gt_classes],
                'predicted_classes': [class_names[int(cls)] for cls in pred_classes],
                'error_type': 'Class mismatch'
            })
            if img_path not in failure_images:
                failure_images.append(img_path)

# Show failure statistics
print(f"Found {len(failure_cases)} failure cases out of {len(test_image_paths[:50])} analyzed test images.")
failure_df = pd.DataFrame(failure_cases)
if not failure_df.empty:
    display(failure_df)

In [ ]:
# Visualize some failure cases
if failure_images:
    fig, axes = plt.subplots(min(len(failure_images), 3), 2, figsize=(14, 12))
    axes = axes.flatten() if len(failure_images) > 1 else [axes]
    
    for i, img_path in enumerate(failure_images[:min(len(failure_images), 3)]):
        # Original image with ground truth
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        # Plot ground truth boxes
        label_path = test_dir / 'labels' / (img_path.stem + '.txt')
        axes[i*2].imshow(img)
        axes[i*2].set_title(f"Ground Truth: {img_path.name}")
        
        with open(label_path, 'r') as f:
            for line in f:
                cls_id, x_center, y_center, width, height = map(float, line.strip().split())
                x1 = int((x_center - width/2) * w)
                y1 = int((y_center - height/2) * h)
                x2 = int((x_center + width/2) * w)
                y2 = int((y_center + height/2) * h)
                
                # Draw rectangle
                rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='green', linewidth=2)
                axes[i*2].add_patch(rect)
                
                # Add label
                axes[i*2].text(x1, y1-5, class_names[int(cls_id)], color='white', 
                               backgroundcolor='green', fontsize=8)
        
        # Predictions
        results = model(img_path, conf=0.25)
        axes[i*2+1].imshow(img)
        axes[i*2+1].set_title(f"Predictions: {img_path.name}")
        
        for box in results[0].boxes.xyxy:
            x1, y1, x2, y2 = box.tolist()
            cls_id = int(results[0].boxes.cls[results[0].boxes.xyxy.tolist().index(box.tolist())])
            conf = float(results[0].boxes.conf[results[0].boxes.xyxy.tolist().index(box.tolist())])
            
            # Draw rectangle
            rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='red', linewidth=2)
            axes[i*2+1].add_patch(rect)
            
            # Add label
            axes[i*2+1].text(x1, y1-5, f"{class_names[cls_id]} {conf:.2f}", color='white', 
                           backgroundcolor='red', fontsize=8)
    
    plt.tight_layout()
    plt.savefig('../backend/model/failure_cases.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No failure cases found in the analyzed test images.")

## 8. Export Model

Finally, let's export the trained model for deployment.

In [ ]:
# Export the model to ONNX format for deployment
model.export(format='onnx')

# Also save PyTorch model format
model_path = '../backend/model/yolov8_weights.pt'
model.save(model_path)

print(f"Model saved to {model_path}")

In [ ]:
# Save metrics to JSON for frontend consumption
import json

# Prepare metrics data
metrics_data = {
    "overallMAP": float(test_results.box.map50),
    "classMetrics": class_metrics,
    "confusionMatrix": confusion_matrix.tolist() if isinstance(confusion_matrix, np.ndarray) else None,
    "failureCases": [failure['image'] for failure in failure_cases] if failure_cases else None
}

# Save to JSON file
metrics_json_path = '../backend/model/metrics.json'
with open(metrics_json_path, 'w') as f:
    json.dump(metrics_data, f, indent=2)

print(f"Metrics saved to {metrics_json_path}")

## 9. Summary

In this notebook, we've:
1. Set up the YOLOv8 environment
2. Prepared and validated our space station object detection dataset
3. Trained a custom YOLOv8 model for detecting toolboxes, oxygen tanks, and fire extinguishers
4. Evaluated performance using precision, recall, F1-score, and mAP metrics
5. Analyzed failure cases to understand model limitations
6. Exported the model for deployment in our application

The model can now be used in the backend service to detect objects in space station images uploaded by users.